In [ ]:
# For PowerBi


output_path = "consumer360_powerbi.xlsx"

rfm_export = rfm_clv.reset_index() if "CustomerID" not in rfm_clv.columns \
             else rfm_clv.copy()

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # Sheet 1 — Full RFM + CLV table
    rfm_export.to_excel(writer, sheet_name="RFM_CLV_Segments", index=False)

    # Sheet 2 — Segment summary
    seg_summary = (rfm_clv.groupby("segment")
                   .agg(customers        = ("monetary", "count"),
                        avg_recency      = ("recency",  "mean"),
                        avg_frequency    = ("frequency","mean"),
                        avg_monetary     = ("monetary", "mean"),
                        total_revenue    = ("monetary", "sum"),
                        avg_clv_adjusted = ("clv_12m_adjusted","mean"))
                   .round(2)
                   .sort_values("total_revenue", ascending=False)
                   .reset_index())
    seg_summary.to_excel(writer, sheet_name="Segment_Summary", index=False)

    # Sheet 3 — Monthly trend
    monthly.to_excel(writer, sheet_name="Monthly_Trend", index=False)

    # Sheet 4 — Top products
    top_products.to_excel(writer, sheet_name="Top_Products", index=False)

    # Sheet 5 — Revenue by country
    country_rev.to_excel(writer, sheet_name="Revenue_By_Country", index=False)

    # Sheet 6 — Cohort retention (long format)
    cohort_data.to_excel(writer, sheet_name="Cohort_Retention", index=False)

    # Sheet 7 — Association rules
    if len(rules_deduped) > 0:
        rules_deduped[["antecedents_str","consequents_str",
                       "support","confidence","lift"]]\
            .to_excel(writer, sheet_name="Market_Basket_Rules", index=False)

    # Sheet 8 — High churn risk customers
    high_risk = rfm_clv[rfm_clv["churn_risk"] == "High Risk"]\
                .sort_values("monetary", ascending=False)
    high_risk_export = high_risk.reset_index() \
        if "CustomerID" not in high_risk.columns else high_risk
    high_risk_export.to_excel(writer, sheet_name="High_Churn_Risk",
                              index=False)

    # Sheet 9 — CLV tiers
    clv_tier_summary = (rfm_clv.groupby("clv_tier")
                        .agg(customers=("clv_12m_adjusted","count"),
                             avg_clv  =("clv_12m_adjusted","mean"),
                             total_clv=("clv_12m_adjusted","sum"))
                        .round(2).reset_index())
    clv_tier_summary.to_excel(writer, sheet_name="CLV_Tiers", index=False)

print(f"  {output_path} saved — 9 sheets ready for Power BI")
print("    Sheets: RFM_CLV_Segments | Segment_Summary | Monthly_Trend |")
print("            Top_Products | Revenue_By_Country | Cohort_Retention |")
print("            Market_Basket_Rules | High_Churn_Risk | CLV_Tiers")

